In [ ]:
import pandas as pd
import numpy as np
import hashlib#hashing

#crear dataset 
np.random.seed(42)
datos_sensibles={
    
    'Cliente_id':[101,102,103,104,105],
    'Nombres':['Carlos Roberto Mendoza','Roy Geovany Umaña','Lionel Andres Messi','Cristiano Ronaldo Silva','Maria Jose Torres',],
    'Tarjeta_credito':['5345-5435-5435-5345','6464-3427-6546-6544','6343-2341-4234-4334','5436-7654-2342-6456','7565-4533-6547-4565'],
    'Salario':[2500.00,4200.00,1800.00,3100.00,5000.00],
    
}

df_pii=pd.DataFrame(datos_sensibles)
print("Datos Con riesgo")
print(df_pii)
print()

#hasshing
#convierte el texto en bytes, convierte el hash en una cadena hexadecimal, solo toma los primeros 12 caracteres
def anonimizar_texto(texto):
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()[:12]

df_anonimizado=df_pii.copy()
df_anonimizado['nombre_anonimizado']=df_anonimizado['Nombres'].apply(anonimizar_texto)
#eliminar columna de nombre
df_anonimizado.drop(columns=['Nombres'],inplace=True)#inplace=True modifica directamente el DataFrame anonimizado y guarda los cambios en el.


print("Nombre Anonimizado")
print(df_anonimizado)
print()


#Enmascarar la tarjeta de credito
def enmascarar_tarjeta(numero_tc):
    #aqui recibe el numero de tarjeta como texto y lo divide usando '-' como separador
    partes=numero_tc.split('-')
    #reemplaza los primero 12 digitos por 'X' y conserva los ultimos 4 digitos
    return f"XXXX-XXXX-XXXX-{partes[-1]}"#obtiene el ultimo elemento de la lista

df_anonimizado['Tarjeta_enmascarada']=df_anonimizado['Tarjeta_credito'].apply(enmascarar_tarjeta)
df_anonimizado.drop(columns=['Tarjeta_credito'],inplace=True)
print("Enmascarar Tarjeta de credito")
print(df_anonimizado)


#perturbacion

#genera un numero aleatorio por cada fila siguiendo una distriucion normal
#0 es la media y 50 es la desviacion estandar del ruido.
ruido=np.random.normal(0,50,size=len(df_anonimizado))

#Suma del ruido aleatorio al salario principal para ocultar su valor real y redondea el resultado a 2 decimales.
#Perturbacion = dato original + ruido aleatorio = dato anonimizado.
df_anonimizado['Salario_anonimizado']=np.round(df_anonimizado['Salario']+ruido,2)
df_anonimizado.drop(columns=['Salario'],inplace=True)
print()
print("Salario con ruido")
print(df_anonimizado)


Datos Con riesgo
   Cliente_id                  Nombres      Tarjeta_credito  Salario
0         101   Carlos Roberto Mendoza  5345-5435-5435-5345   2500.0
1         102        Roy Geovany Umaña  6464-3427-6546-6544   4200.0
2         103      Lionel Andres Messi  6343-2341-4234-4334   1800.0
3         104  Cristiano Ronaldo Silva  5436-7654-2342-6456   3100.0
4         105        Maria Jose Torres  7565-4533-6547-4565   5000.0

Nombre Anonimizado
   Cliente_id      Tarjeta_credito  Salario nombre_anonimizado
0         101  5345-5435-5435-5345   2500.0       1660ce8cb081
1         102  6464-3427-6546-6544   4200.0       844473d4b394
2         103  6343-2341-4234-4334   1800.0       17d3624ef4f2
3         104  5436-7654-2342-6456   3100.0       a311bf7041ed
4         105  7565-4533-6547-4565   5000.0       c91681d53909

Enmascarar Tarjeta de credito
   Cliente_id  Salario nombre_anonimizado  Tarjeta_enmascarada
0         101   2500.0       1660ce8cb081  XXXX-XXXX-XXXX-5345
1         102 